In [1]:
# third-party
import pandas as pd
import scipy
import pickle
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# local
from respiratoryanalysis.get_data import transform_overview_on_target, transform_overview_on_overall
from respiratoryanalysis.performance import get_breath_parameters_performance, get_breath_parameters_performance_display
from respiratoryanalysis.delay_performance import normality_test
from respiratoryanalysis.get_data import get_participant_ids
from respiratoryanalysis.performance import bland_altman_plot, get_relative_errors
from respiratoryanalysis.get_data import get_data_by_id_activity, list_activities
from respiratoryanalysis.constants import CATEGORICAL_PALETTE


In [2]:
with open('../results/results.pickle', 'rb') as file:
    overview = pickle.load(file)

overview_middle = {}
for id in overview.keys():
    overview_middle[id] = {key: value for key, value in overview[id].items() if key in ['SNBm', 'UALm', 'UARm']}

# remove middle activities
for id in overview.keys():
    del overview[id]["SNBm"]
    del overview[id]["UALm"]
    del overview[id]["UARm"]

In [3]:
acquisition_folderpath = "/Users/anasofiacc/dev/MAG-PZT-Dataset/Data"
activity_list = list_activities(acquisition_folderpath)
data, _ = get_data_by_id_activity(acquisition_folderpath, list(overview.keys()), activity_list)

Getting data for participants...
 --------- 7OYX ---------------
 --------- NO15 ---------------
 --------- G8B7 ---------------
 --------- EPE2 ---------------
 --------- HAK8 ---------------
 --------- 1BST ---------------
 --------- 83J1 ---------------
 --------- QMQ7 ---------------
Data for participant QMQ7 and activity MIXB not found. Skipping...
 --------- 9TUL ---------------
 --------- FTD7 ---------------
 --------- Y6O3 ---------------
 --------- 2QWT ---------------
 --------- F9AF ---------------
 --------- P4W9 ---------------
 --------- W8Z9 ---------------
 --------- D4GQ ---------------


In [4]:
def plot_resp_param(data, overview, id, activity, sensor, sampling_freq=100):

    sensor_dict = {"MAG": "mag", "PZT": "pzt"}

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # plot sensor and airflow signals
    fig.add_trace(go.Scatter(x=np.linspace(1, len(data[id][activity][sensor_dict[sensor]]),len(data[id][activity][sensor_dict[sensor]]))/sampling_freq, 
                             y=data[id][activity][sensor_dict[sensor]], 
                             name=sensor))
    fig.add_trace(go.Scatter(x=np.linspace(1, len(data[id][activity]['airflow']),len(data[id][activity]['airflow']))/sampling_freq,
                             y=data[id][activity]['airflow'], 
                            name="Airflow", line={"color":"white"}), secondary_y=True)

    # add FR events
    peak_valley = {"TP_i": "valleys", "TP_e": "peaks", "FN_i": "valleys", "FN_e": "peaks", "FP_i": "valleys", "FP_e": "peaks"}

    for event in ["TP_i", "TP_e"]:
        if event == "TP_i":
            symbol = "triangle-up"
        else:
            symbol = "triangle-down"
        fig.add_trace(go.Scatter(x=np.array(overview[id][activity][sensor][event])/sampling_freq, y=data[id][activity][sensor_dict[sensor]][overview[id][activity][sensor][event]],
                            mode="markers",
                            marker = {
                                'symbol': symbol,
                                'size': 10
                            },
                            name=f"TP {peak_valley[event]}"))
        
        fig.add_trace(go.Scatter(x=np.array(overview[id][activity]["Airflow"][peak_valley[event]])/sampling_freq, y=data[id][activity]['airflow'][overview[id][activity]["Airflow"][peak_valley[event]]],
                        mode="markers",
                        marker = {
                                'color': 'white',
                                'symbol': symbol,
                                'size': 10
                            },
                        name=f"ref {peak_valley[event]}",), secondary_y=True)
    
    for event in ["FN_i", "FN_e"]:
        fig.add_trace(go.Scatter(x=np.array(overview[id][activity][sensor][event])/sampling_freq, y=data[id][activity][sensor_dict[sensor]][overview[id][activity][sensor][event]],
                            mode="markers",
                            name=f"FN {peak_valley[event]}"))
        
    for event in ["FP_i", "FP_e"]:
        fig.add_trace(go.Scatter(x=np.array(overview[id][activity][sensor][event])/sampling_freq, y=data[id][activity][sensor_dict[sensor]][overview[id][activity][sensor][event]],
                            mode="markers",
                            name=f"FP {peak_valley[event]}"))


    TP_peaks_tuple = [(event, True, "peak") for event in overview[id][activity][sensor]["TP_e"]]
    TP_valleys_tuple = [(event, True, "valley") for event in overview[id][activity][sensor]["TP_i"]]
    FN_peaks_tuple = [(event, False, "peak") for event in overview[id][activity][sensor]["FN_e"]]
    FN_valleys_tuple = [(event, False, "valley") for event in overview[id][activity][sensor]["FN_i"]]
    extrema = sorted(TP_peaks_tuple + TP_valleys_tuple + FN_peaks_tuple + FN_valleys_tuple)

    
    extrema_pairs_peaks = np.asarray([
        (extrema[i][0], extrema[i+1][0],
            all((extrema[i][1], extrema[i+1][1])))
        for i in range(0, len(extrema)-1)
        if extrema[i][2] == "peak" and extrema[i+1][2] == "valley"
    ])

    extrema_pairs_valleys = np.asarray([
        (extrema[i][0], extrema[i+1][0],
            all((extrema[i][1], extrema[i+1][1])))
        for i in range(0, len(extrema)-1)
        if extrema[i][2] == "valley" and extrema[i+1][2] == "peak"
    ])

    extrema_pairs_peaks = extrema_pairs_peaks[extrema_pairs_peaks[:,-1] != 0]
    extrema_pairs_valleys = extrema_pairs_valleys[extrema_pairs_valleys[:,-1] != 0]

    for i,ind in enumerate(extrema_pairs_peaks):
        try:
            exp_duration = overview[id][activity][sensor]["tE (s)"][i]
            fig.add_vrect(x0=ind[0]/sampling_freq, x1=ind[0]/sampling_freq+exp_duration, fillcolor=CATEGORICAL_PALETTE[0], 
                            opacity=0.2, line_width=0.2, line_color='white', annotation_text="exp", annotation_position="top left",)
        except Exception as e:
            print(e)

    for i, ind in enumerate(extrema_pairs_valleys):  
        try:
            insp_duration = overview[id][activity][sensor]["tI (s)"][i]
            fig.add_vrect(x0=ind[0]/sampling_freq, x1=ind[0]/sampling_freq+insp_duration, fillcolor=CATEGORICAL_PALETTE[1], 
                            opacity=0.2, line_width=0.2, line_color='white', annotation_text="insp", annotation_position="bottom left",)
        except Exception as e:
            print(e)
                
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)
    fig.show()

    print(f'sensor: {overview[id][activity][sensor]["tI (s)"]} | airflow: {overview[id][activity][sensor]["tI airflow (s)"]}')
    

In [5]:
plot_resp_param(data, overview, id="2QWT", activity="SNB", sensor="MAG")


sensor: [4.1  3.26 4.01 4.25 3.54 4.12 3.77] | airflow: [3.89 3.32 3.78 4.3  3.42 4.08 3.59]


In [6]:
plot_resp_param(data, overview, id="2QWT", activity="SNB", sensor="PZT")

sensor: [6.63 5.48 5.51 7.18 5.09 7.56 1.75] | airflow: [3.89 3.32 3.78 4.3  3.42 4.08 3.59]


### Activity-specific only RP analysis

In [7]:
# Transform overview from participant specific into all participants

overview_all_participants = transform_overview_on_target(overview, target="Activity")
overview_all_participants.keys()

dict_keys(['SNB', 'SGB', 'MIXB', 'STNB', 'MCH', 'SQT', 'AAL', 'AAR', 'ALL', 'ALR', 'UAL', 'UAR', 'SE', 'SS', 'TR'])

In [8]:
performance_all_participants = get_breath_parameters_performance_display(overview_all_participants, target="Activity")
performance_all_participants[performance_all_participants["Sensor"]=="MAG"]

,Activity,Sensor,MAE Ti (s),MRE Ti (%),MAE Te (s),MRE Te (%),MAE Tb (s),MRE Tb (%)
0,SNB,MAG,0.07 $\pm$ 0.11,3.18 $\pm$ 4.80,0.07 $\pm$ 0.11,2.75 $\pm$ 4.22,0.06 $\pm$ 0.10,1.34 $\pm$ 2.18
2,SGB,MAG,0.49 $\pm$ 0.54,10.89 $\pm$ 12.36,0.49 $\pm$ 0.56,8.73 $\pm$ 10.03,0.34 $\pm$ 0.55,3.39 $\pm$ 5.54
4,MIXB,MAG,0.11 $\pm$ 0.14,3.93 $\pm$ 4.98,0.11 $\pm$ 0.15,4.09 $\pm$ 5.24,0.09 $\pm$ 0.12,1.62 $\pm$ 1.93
6,STNB,MAG,0.10 $\pm$ 0.18,4.17 $\pm$ 6.80,0.10 $\pm$ 0.16,3.76 $\pm$ 5.36,0.08 $\pm$ 0.14,1.71 $\pm$ 3.17
8,MCH,MAG,0.20 $\pm$ 0.28,9.01 $\pm$ 13.17,0.22 $\pm$ 0.40,8.56 $\pm$ 11.27,0.19 $\pm$ 0.35,4.02 $\pm$ 7.21
10,SQT,MAG,0.18 $\pm$ 0.23,8.50 $\pm$ 8.61,0.15 $\pm$ 0.21,7.55 $\pm$ 8.46,0.17 $\pm$ 0.25,4.37 $\pm$ 5.19
12,AAL,MAG,0.14 $\pm$ 0.16,6.95 $\pm$ 6.98,0.15 $\pm$ 0.18,7.51 $\pm$ 8.49,0.15 $\pm$ 0.18,3.92 $\pm$ 5.12
14,AAR,MAG,0.10 $\pm$ 0.14,5.33 $\pm$ 7.16,0.11 $\pm$ 0.14,5.58 $\pm$ 6.40,0.12 $\pm$ 0.20,3.07 $\pm$ 5.00
16,ALL,MAG,0.13 $\pm$ 0.15,6.85 $\pm$ 7.07,0.11 $\pm$ 0.11,6.74 $\pm$ 6.09,0.13 $\pm$ 0.15,3.58 $\pm$ 3.65
18,ALR,MAG,0.17 $\pm$ 0.20,9.64 $\pm$ 10.73,0.15 $\pm$ 0.22,8.20 $\pm$ 9.41,0.16 $\pm$ 0.16,4.66 $\pm$ 4.68


In [9]:
from scikit_posthocs import posthoc_dunn

for metric in ["tI", "tE", "tB"]:
    for device in ["MAG", "PZT"]:
        relative_errors_by_activity = {}
        for activity in overview_all_participants.keys():
            relative_errors_by_activity[activity] = get_relative_errors(overview_all_participants[activity][device][f"{metric} airflow (s)"], overview_all_participants[activity][device][f"{metric} (s)"])
            # normality_test(relative_errors, sensor=device, type=f"relative error {metric}")
        
        h_statistic, p_value = scipy.stats.kruskal(
            relative_errors_by_activity["SNB"],
            relative_errors_by_activity["SGB"],
            relative_errors_by_activity["MIXB"],
            relative_errors_by_activity["STNB"],
            relative_errors_by_activity["MCH"],
            relative_errors_by_activity["SQT"],
            relative_errors_by_activity["AAL"],
            relative_errors_by_activity["AAR"],
            relative_errors_by_activity["ALL"],
            relative_errors_by_activity["ALR"],
            relative_errors_by_activity["UAL"],
            relative_errors_by_activity["UAR"],
            relative_errors_by_activity["SE"],
            relative_errors_by_activity["SS"],
            relative_errors_by_activity["TR"],
        )
        print("\n")
        print(f"Kruskal-Wallis H-test test for {metric} {device}: H()={h_statistic:.1f} p={p_value:.4f}")
        
        if p_value < 0.05:
            relative_errors_list = [relative_errors_by_activity[activity].tolist() for activity in relative_errors_by_activity]
            #print(np.argwhere(posthoc_dunn(relative_errors_list, p_adjust="bonferroni") < 0.05))
            dunn = posthoc_dunn(relative_errors_list, p_adjust="bonferroni")

            for i, activity in enumerate(overview_all_participants.keys()):
                aux = np.argwhere(dunn.iloc[i, :] < 0.05)
                print(f"{activity}: {[list(overview_all_participants.keys())[a] for a in aux.flatten()]}")                
    print("\n")





ModuleNotFoundError: No module named 'scikit_posthocs'

### Overall RP analysis

In [10]:
# Transform overview from participant specific into all participants
# overview_all = transform_overview_on_overall(overview)
# overview_all.keys()
overview_all_activities = transform_overview_on_target(overview, target="ID")
overview_all_activities.keys()

dict_keys(['7OYX', 'NO15', 'G8B7', 'EPE2', 'HAK8', '1BST', '83J1', 'QMQ7', '9TUL', 'FTD7', 'Y6O3', '2QWT', 'F9AF', 'P4W9', 'W8Z9', 'D4GQ'])

In [11]:
get_breath_parameters_performance_display(overview_all_activities, target="ID")

,ID,Sensor,MAE Ti (s),MRE Ti (%),MAE Te (s),MRE Te (%),MAE Tb (s),MRE Tb (%)
0,7OYX,MAG,0.20 $\pm$ 0.30,7.50 $\pm$ 9.66,0.22 $\pm$ 0.36,9.50 $\pm$ 15.83,0.21 $\pm$ 0.30,4.26 $\pm$ 5.88
1,7OYX,PZT,0.69 $\pm$ 0.88,26.75 $\pm$ 32.98,0.64 $\pm$ 0.81,26.29 $\pm$ 32.57,0.69 $\pm$ 0.87,13.92 $\pm$ 17.05
2,NO15,MAG,0.15 $\pm$ 0.20,6.82 $\pm$ 7.95,0.14 $\pm$ 0.18,6.07 $\pm$ 6.43,0.16 $\pm$ 0.20,3.57 $\pm$ 4.01
3,NO15,PZT,0.35 $\pm$ 0.42,15.12 $\pm$ 16.12,0.33 $\pm$ 0.50,13.82 $\pm$ 18.10,0.30 $\pm$ 0.38,6.98 $\pm$ 8.20
4,G8B7,MAG,0.30 $\pm$ 0.41,16.06 $\pm$ 16.52,0.30 $\pm$ 0.40,16.33 $\pm$ 16.72,0.26 $\pm$ 0.41,7.40 $\pm$ 8.55
5,G8B7,PZT,0.14 $\pm$ 0.20,8.81 $\pm$ 9.89,0.16 $\pm$ 0.27,9.30 $\pm$ 9.98,0.19 $\pm$ 0.37,5.43 $\pm$ 6.32
6,EPE2,MAG,0.21 $\pm$ 0.25,8.52 $\pm$ 7.96,0.19 $\pm$ 0.25,7.73 $\pm$ 8.08,0.17 $\pm$ 0.22,3.72 $\pm$ 4.22
7,EPE2,PZT,0.58 $\pm$ 0.76,23.99 $\pm$ 29.11,0.54 $\pm$ 0.70,21.22 $\pm$ 23.30,0.36 $\pm$ 0.50,7.69 $\pm$ 9.80
8,HAK8,MAG,0.19 $\pm$ 0.37,8.50 $\pm$ 14.69,0.18 $\pm$ 0.33,8.14 $\pm$ 10.89,0.19 $\pm$ 0.31,4.60 $\pm$ 5.99
9,HAK8,PZT,0.53 $\pm$ 0.72,23.59 $\pm$ 24.02,0.63 $\pm$ 0.81,28.09 $\pm$ 29.01,0.55 $\pm$ 0.61,14.08 $\pm$ 15.54


In [12]:
breath_parameters = get_breath_parameters_performance(overview_all_activities, target="ID")
breath_parameters

,ID,Sensor,MAE Ti (s),MRE Ti (%),MAE Te (s),MRE Te (%),MAE Tb (s),MRE Tb (%)
0,7OYX,MAG,0.195847,7.503441,0.223978,9.497451,0.210843,4.258743
1,7OYX,PZT,0.685312,26.754465,0.638629,26.285172,0.691140,13.916826
2,NO15,MAG,0.151005,6.820145,0.142283,6.065860,0.156791,3.566395
3,NO15,PZT,0.345673,15.117805,0.331165,13.815415,0.300754,6.982527
4,G8B7,MAG,0.295355,16.058885,0.299898,16.327703,0.260602,7.402766
5,G8B7,PZT,0.144202,8.814860,0.161613,9.295768,0.187031,5.425906
6,EPE2,MAG,0.209382,8.521525,0.191854,7.728943,0.172807,3.724164
7,EPE2,PZT,0.578084,23.994512,0.542000,21.221717,0.357933,7.689451
8,HAK8,MAG,0.186561,8.502910,0.175622,8.136734,0.192757,4.601313
9,HAK8,PZT,0.529044,23.585371,0.628467,28.087407,0.548966,14.079670


In [22]:
import plotly.graph_objects as go

for metric in ["Ti", "Te", "Tb"]:
    rel_error_mag = breath_parameters[breath_parameters["Sensor"]=="MAG"][f"MRE {metric} (%)"]
    rel_error_pzt = breath_parameters[breath_parameters["Sensor"]=="PZT"][f"MRE {metric} (%)"]

    print(f"{metric}: W({len(rel_error_mag) + len(rel_error_pzt)}) {scipy.stats.wilcoxon(rel_error_mag, rel_error_pzt)}")

    fig = go.Figure()
    fig.add_trace(go.Violin(y=rel_error_mag, name="MAG"))
    fig.add_trace(go.Violin(y=rel_error_pzt, name="PZT"))
# 
    #fig.show()

    # normality_test(rel_error_mag, sensor="MAG", type=f"relative error {metric}")
    # normality_test(rel_error_pzt, sensor="PZT", type=f"relative error {metric}")


Ti: W(32) WilcoxonResult(statistic=np.float64(7.0), pvalue=np.float64(0.000579833984375))
Te: W(32) WilcoxonResult(statistic=np.float64(7.0), pvalue=np.float64(0.000579833984375))
Tb: W(32) WilcoxonResult(statistic=np.float64(1.0), pvalue=np.float64(6.103515625e-05))


### Bias and variability analysis

In [ ]:
for id in overview.keys():
    for activity in overview[id].keys():
        for sensor in ["MAG", "PZT"]:
            if any(overview[id][activity][sensor]["tE (s)"] > 10):
                print(f'{overview[id][activity][sensor]["tE (s)"]} ({id} - {activity} - {sensor})')

[ 6.79  3.08  6.21 11.4   4.14  6.32] (QMQ7 - SGB - PZT)
[10.76  7.98  5.14] (Y6O3 - SGB - PZT)
[ 1.53  1.38  5.13 10.36  3.05  3.25  2.84  1.8 ] (2QWT - MIXB - PZT)


In [ ]:
from performance import get_bias_variability

get_bias_variability(overview_all_participants, target="Activity", metric="tI", relative_error=False)
get_bias_variability(overview_all, target="overview", metric="tI", relative_error=False)

,Sensor,Bias (s),Variability (s)
0,MAG,0.058,0.565
1,PZT,0.014,0.826


In [ ]:
overview_all_except =  transform_overview_on_overall(overview, ignore_key="SGB")

In [ ]:
activity = "SS"
for metric in ["tI", "tE", "tB"]:
        bland_altman_plot(overview_all_except["MAG"][f"{metric} (s)"], overview_all_except["MAG"][f"{metric} airflow (s)"], overview_all_except["PZT"][f"{metric} (s)"], overview_all_except["PZT"][f"{metric} airflow (s)"], metric)
        #bland_altman_plot(overview_all_participants[activity]["MAG"][f"{metric} (s)"], overview_all_participants[activity]["MAG"][f"{metric} airflow (s)"], overview_all_participants[activity]["PZT"][f"{metric} (s)"], overview_all_participants[activity]["PZT"][f"{metric} airflow (s)"], metric, activity)


MAG | mean: 0.051, lloa: -0.98, uloa: 1.08 
PZT | mean: 0.026, lloa: -1.52, uloa: 1.57 


MAG | mean: -0.059, lloa: -1.11, uloa: 0.99 
PZT | mean: -0.046, lloa: -1.66, uloa: 1.57 


MAG | mean: -0.005, lloa: -0.98, uloa: 0.97 
PZT | mean: -0.021, lloa: -1.54, uloa: 1.50 
